# Debug Scraper Detik Jatim

Local HTML first, then optional live list/article checks. Output list CSV: `csv/detik_jatim_list_debug.csv`.

In [1]:
from pathlib import Path
import sys
import importlib
from urllib.parse import urljoin

import pandas as pd
from bs4 import BeautifulSoup

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scrapers').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scrapers.common import cutoff_date, parse_date, records_to_df
import scrapers.detikjatim as scraper
scraper = importlib.reload(scraper)

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 160)

SAMPLE_PATH = PROJECT_ROOT / 'html_samples/detikjatim.html'
OUTPUT_PATH = PROJECT_ROOT / 'csv/detik_jatim_list_debug.csv'
MAX_PAGES = 200
TITLE_LIMIT = 90
cutoff = cutoff_date()

print('project:', PROJECT_ROOT)
print('module:', scraper.__file__)
print('sample:', SAMPLE_PATH)
print('cutoff:', cutoff.date())


project: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project
module: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/scrapers/detikjatim.py
sample: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/html_samples/detikjatim.html
cutoff: 2026-02-28


In [2]:

def short_title(value, limit=TITLE_LIMIT):
    text = '' if pd.isna(value) else str(value).strip()
    return text if len(text) <= limit else text[: limit - 3] + '...'


def ensure_debug_columns(df):
    df = df.copy()
    if 'page_num' not in df.columns:
        df['page_num'] = pd.NA
    if 'published_date' not in df.columns:
        df['published_date'] = None
    df['published_dt'] = df['published_date'].apply(parse_date)
    return df


def print_debug_rows(df):
    for _, row in df.iterrows():
        page = row.get('page_num')
        page_text = '---' if pd.isna(page) else f'{int(page):03d}'
        date_text = row.get('published_date')
        date_text = 'None' if pd.isna(date_text) else str(date_text)
        print(f"page={page_text} | date={date_text} | title={short_title(row.get('title'))}")


def audit_list(df):
    df = ensure_debug_columns(df)
    print('total rows:', len(df))
    if len(df) == 0:
        print('No rows found.')
        return df
    print('first page:', df['page_num'].dropna().min() if df['page_num'].notna().any() else None)
    print('last page:', df['page_num'].dropna().max() if df['page_num'].notna().any() else None)
    print('newest date:', df['published_dt'].max())
    print('oldest date:', df['published_dt'].min())
    print('cutoff date:', cutoff)
    print('reached cutoff:', bool((df['published_dt'].dropna() < cutoff).any()))
    print('null parsed dates:', int(df['published_dt'].isna().sum()))

    print('\nrows per month:')
    display(
        df.assign(month=df['published_dt'].dt.to_period('M').astype(str))
        .groupby('month', dropna=False)
        .size()
        .reset_index(name='count')
    )

    print('\nrows per page:')
    display(
        df.groupby('page_num', dropna=False)
        .agg(rows=('url', 'count'), newest=('published_dt', 'max'), oldest=('published_dt', 'min'))
        .reset_index()
        .tail(80)
    )
    return df


In [3]:

def parse_local_list(html_text):
    soup = BeautifulSoup(html_text, 'html.parser')
    rows = []
    for card in soup.select('div.list-berita article, article'):
        link_tag = card.select_one('a[href]')
        title_tag = card.select_one('h2.title')
        date_tag = card.select_one('.date')
        excerpt_tag = card.select_one('p')
        image_tag = card.select_one('img')
        if not link_tag or not title_tag:
            continue
        rows.append({
            'page_num': 1,
            'list_page_url': scraper.BASE_URL,
            'title': title_tag.get_text(' ', strip=True),
            'url': urljoin(scraper.BASE_URL, link_tag['href']),
            'published_date': scraper.clean_list_date(date_tag.get_text(' ', strip=True) if date_tag else None),
            'excerpt': excerpt_tag.get_text(' ', strip=True) if excerpt_tag else None,
            'image_url': (image_tag.get('data-src') or image_tag.get('src')) if image_tag else None,
        })
    return records_to_df(rows)


## Local HTML list parse

In [4]:
html_text = SAMPLE_PATH.read_text(errors='replace')
local_df = parse_local_list(html_text)
local_df = ensure_debug_columns(local_df)
print_debug_rows(local_df)
local_df = audit_list(local_df)


page=001 | date=Selasa, 16 Jun 2026 13:00 WIB | title=Nekat Lewat Jalur Ilegal, 13 Pendaki Gunung Semeru Diamankan
page=001 | date=Selasa, 12 Mei 2026 19:00 WIB | title=Pesona Desa Wisata Peniwen Malang, Wisata Religi Kenaikan Yesus Kristus
page=001 | date=Senin, 20 Apr 2026 22:55 WIB | title=Bupati Malang Lantik Anak Kandung Jadi Kadis LH, Ini Penjelasan BKPSDM
page=001 | date=Jumat, 17 Apr 2026 11:50 WIB | title=Kata PDIP Soal Bupati Malang Lantik Anak Kandung Jadi Kepala Dinas LH
page=001 | date=Rabu, 08 Apr 2026 18:12 WIB | title=Longsor Terjang Dusun Sengon Malang, Jalan Sempat Lumpuh Total
page=001 | date=Senin, 30 Mar 2026 15:58 WIB | title=Gus Ipul Ajak Kades Se-Malang Hidupkan Puskesos dan Mutakhirkan Data
page=001 | date=Sabtu, 28 Mar 2026 19:15 WIB | title=Hujan Angin di Malang Bikin Pohon Tumbang Timpa Rumah dan Warung
page=001 | date=Sabtu, 21 Mar 2026 15:40 WIB | title=Lebaran Pertama, Ribuan Kendaraan Serbu Malang-Batu Lewat Exit Tol Singosari
page=001 | date=Kamis, 05 M

,month,count
0,2026-02,1
1,2026-03,4
2,2026-04,3
3,2026-05,1
4,2026-06,1



rows per page:


,page_num,rows,newest,oldest
0,1,10,2026-06-16 13:00:00,2026-02-14 10:55:00


## Live list scrape

In [5]:
live_df = scraper.scrape_list(cutoff, max_pages=MAX_PAGES)
live_df = ensure_debug_columns(live_df)
print_debug_rows(live_df)
live_df = audit_list(live_df)


Scraping Detik Jatim page 1: https://www.detik.com/tag/kabupaten-malang
Scraping Detik Jatim page 2: https://www.detik.com/tag/kabupaten-malang/2
Gagal buka Detik Jatim page 2: 404 Client Error: Not Found for url: https://www.detik.com/tag/kabupaten-malang/2
page=001 | date=Selasa, 16 Jun 2026 13:00 WIB | title=Nekat Lewat Jalur Ilegal, 13 Pendaki Gunung Semeru Diamankan
page=001 | date=Selasa, 12 Mei 2026 19:00 WIB | title=Pesona Desa Wisata Peniwen Malang, Wisata Religi Kenaikan Yesus Kristus
page=001 | date=Senin, 20 Apr 2026 22:55 WIB | title=Bupati Malang Lantik Anak Kandung Jadi Kadis LH, Ini Penjelasan BKPSDM
page=001 | date=Jumat, 17 Apr 2026 11:50 WIB | title=Kata PDIP Soal Bupati Malang Lantik Anak Kandung Jadi Kepala Dinas LH
page=001 | date=Rabu, 08 Apr 2026 18:12 WIB | title=Longsor Terjang Dusun Sengon Malang, Jalan Sempat Lumpuh Total
page=001 | date=Senin, 30 Mar 2026 15:58 WIB | title=Gus Ipul Ajak Kades Se-Malang Hidupkan Puskesos dan Mutakhirkan Data
page=001 | date=

,month,count
0,2026-03,4
1,2026-04,3
2,2026-05,1
3,2026-06,1



rows per page:


,page_num,rows,newest,oldest
0,1,9,2026-06-16 13:00:00,2026-03-05 16:54:00


## Save debug list CSV

In [6]:
df_to_save = live_df if 'live_df' in globals() and len(live_df) else local_df
OUTPUT_PATH.parent.mkdir(exist_ok=True)
df_to_save.to_csv(OUTPUT_PATH, index=False)
print('saved:', OUTPUT_PATH)


saved: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/csv/detik_jatim_list_debug.csv


## Optional article sample check

In [7]:
sample_rows = (live_df if 'live_df' in globals() and len(live_df) else local_df).head(5)
article_rows = []
for index, row in sample_rows.iterrows():
    try:
        article = scraper.extract_article(row)
        article_rows.append(article)
        print(f"[{len(article_rows)}] content_len={len(article.get('content') or '')} | {short_title(article.get('title'))}")
    except Exception as error:
        print(f"failed sample article {index + 1}: {row.get('url')} | {error}")
article_df = pd.DataFrame(article_rows)
display(article_df[['title', 'published_date', 'content', 'url']].head() if len(article_df) else article_df)


[1] content_len=3185 | Nekat Lewat Jalur Ilegal, 13 Pendaki Gunung Semeru Diamankan
[2] content_len=4228 | Pesona Desa Wisata Peniwen Malang, Wisata Religi Kenaikan Yesus Kristus
[3] content_len=2332 | Bupati Malang Lantik Anak Kandung Jadi Kadis LH, Ini Penjelasan BKPSDM
[4] content_len=2957 | Kata PDIP Soal Bupati Malang Lantik Anak Kandung Jadi Kepala Dinas LH
[5] content_len=2449 | Longsor Terjang Dusun Sengon Malang, Jalan Sempat Lumpuh Total


,title,published_date,content,url
0,"Nekat Lewat Jalur Ilegal, 13 Pendaki Gunung Semeru Diamankan",2026-06-16,Malang\n-\nBalai Besar Taman Nasional Bromo Tengger Semeru (BB TNBTS) berhasil mengamankan 13 orang yang melakukan pendakian ilegal ke Gunung Semeru. Belasa...,https://www.detik.com/jatim/wisata/d-8534025/nekat-lewat-jalur-ilegal-13-pendaki-gunung-semeru-diamankan
1,"Pesona Desa Wisata Peniwen Malang, Wisata Religi Kenaikan Yesus Kristus",2026-05-12,Daftar Isi\nLokasi Desa Peniwen Malang\nSejarah Desa Peniwen Malang\nPanorama dan Daya Tarik Desa Peniwen Malang\nHarga Tiket dan Paket Wisata Desa Peniwen ...,https://www.detik.com/jatim/wisata/d-8485618/pesona-desa-wisata-peniwen-malang-wisata-religi-kenaikan-yesus-kristus
2,"Bupati Malang Lantik Anak Kandung Jadi Kadis LH, Ini Penjelasan BKPSDM",2026-04-20,Jakarta\n-\nBadan Kepegawaian dan Pengembangan Sumber Daya Manusia (BKPSDM) Kabupaten Malang buka suara terkait pengangkatan\nAhmad Dzulfikar Nurrahman\nseb...,https://news.detik.com/berita/d-8453548/bupati-malang-lantik-anak-kandung-jadi-kadis-lh-ini-penjelasan-bkpsdm
3,Kata PDIP Soal Bupati Malang Lantik Anak Kandung Jadi Kepala Dinas LH,2026-04-17,"Malang\n-\nDeddy Sitorus, Ketua DPP PDIP menanggapi tentang Bupati Malang M Sanusi yang sedang jadi sorotan karena melantik putra kandungnya, Ahmad Dzulfika...",https://www.detik.com/jatim/berita/d-8448466/kata-pdip-soal-bupati-malang-lantik-anak-kandung-jadi-kepala-dinas-lh
4,"Longsor Terjang Dusun Sengon Malang, Jalan Sempat Lumpuh Total",2026-04-08,"Malang\n-\nHujan deras yang mengguyur wilayah Kabupaten Malang memicu bencana tanah longsor di Desa Dalisodo, Kecamatan Wagir. Akibatnya, akses jalan di Dus...",https://www.detik.com/jatim/berita/d-8435257/longsor-terjang-dusun-sengon-malang-jalan-sempat-lumpuh-total
